[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/benchmarks/Benchmark_Throughput.ipynb)

# Benchmark: Throughput

**How many tokens, sentences, etc. can a Spark NLP pipeline process per second?**

`sparknlp.benchmark.Benchmark.throughput` answers this for *any* fitted pipeline and *any*
data you give it. You don't tell it what to measure -- it looks at what your pipeline actually
produces (tokens, sentences, named entities, ...) and reports a rate for each, automatically.

This notebook builds a small tokenizer pipeline and times it against a real book.

## Setup

Run these cells first on a fresh Colab runtime.

In [3]:
!wget https://setup.johnsnowlabs.com/colab.sh -O - | bash

--2026-08-29 10:57:08--  https://setup.johnsnowlabs.com/colab.sh
Resolving setup.johnsnowlabs.com (setup.johnsnowlabs.com)... 3.86.22.73
Connecting to setup.johnsnowlabs.com (setup.johnsnowlabs.com)|3.86.22.73|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh [following]
--2026-08-29 10:57:08--  https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp/master/scripts/colab_setup.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1483 (1.4K) [text/plain]
Saving to: ‘STDOUT’


-                     0%[                    ]       0  --.-KB/s               
-                   100%[===================>]   1.45K  --.-KB/s    in 0s      



In [4]:
# Current Colab runtimes default to Java 21, which Spark 3.4.x (what the bootstrap above
# installs) isn't compatible with -- Spark's low-level Platform.java reflection breaks on it.
# Switch to Java 17, which Spark 3.4.x does support.
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [5]:
# Current Colab runtimes also default to Python 3.13, which removed the deprecated
# `typing.io` submodule -- but Spark 3.4.x's own source still does `from typing.io import
# BinaryIO`. This patches that one import, both for this notebook process and for the
# separate Python worker subprocesses Spark launches to actually run distributed tasks
# (those load pyspark from its own bundled zip, so both copies need patching).
import zipfile, shutil

site_pkgs = "/usr/local/lib/python3.13/dist-packages"
loose_path = f"{site_pkgs}/pyspark/broadcast.py"
zip_path = f"{site_pkgs}/pyspark/python/lib/pyspark.zip"
OLD = "from typing.io import BinaryIO  # type: ignore[import]"
NEW = "from typing import BinaryIO  # patched for Python 3.13 (typing.io removed)"

with open(loose_path) as f:
    text = f.read()
with open(loose_path, "w") as f:
    f.write(text.replace(OLD, NEW))

tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(zip_path, "r") as zin, zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as zout:
    for item in zin.infolist():
        data = zin.read(item.filename)
        if item.filename == "pyspark/broadcast.py":
            data = data.decode("utf-8").replace(OLD, NEW).encode("utf-8")
        zout.writestr(item, data)
shutil.move(tmp_path, zip_path)
print("Environment patched for this Colab runtime (Java 17, typing.io).")

Environment patched for this Colab runtime (Java 17, typing.io).

In [6]:
import sparknlp
spark = sparknlp.start()
print("Spark NLP version:", sparknlp.version())
print("Apache Spark version:", spark.version)
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from pyspark.ml import Pipeline
from sparknlp.benchmark import Benchmark

Spark NLP version: 6.4.2
Apache Spark version: 3.4.4

## 1. Get some data

We'll use *Pride and Prejudice* (Jane Austen, public domain) from Project Gutenberg, split into
paragraphs so we have many independent rows to measure throughput across. Any `text`-column
DataFrame works here -- swap in your own data.

In [8]:
import urllib.request

url = "https://www.gutenberg.org/files/1342/1342-0.txt"
raw_text = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")

# Trim Project Gutenberg's license header/footer, keep the book's own text.
start = raw_text.find("*** START OF")
end = raw_text.find("*** END OF")
body = raw_text[start:end]

paragraphs = [p.strip().replace("\n", " ") for p in body.split("\n\n")]
paragraphs = [p for p in paragraphs if len(p) > 250 and "Illustration" not in p and not p.isupper()]
print(f"{len(paragraphs)} paragraphs")
print(paragraphs[10][:300])

888 paragraphs
_The Bingleys and the Gardiners and the Lucases, Miss Darcy and Miss de Bourgh, Jane, Wickham, and the rest, must pass without special comment, further than the remark that Charlotte Lucas (her egregious papa, though delightful, is just a little on the thither side of the line between comedy and far

In [9]:
data = spark.createDataFrame([(p,) for p in paragraphs], ["text"]).repartition(4)
data.cache()
print(data.count(), "rows")

888 rows

## 2. Build a pipeline

Any pipeline works -- here's a simple sentence + token one.

In [11]:
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
sentence_detector = SentenceDetector().setInputCols(["document"]).setOutputCol("sentence")
tokenizer = Tokenizer().setInputCols(["sentence"]).setOutputCol("token")

pipeline = Pipeline(stages=[document_assembler, sentence_detector, tokenizer])
pipeline_model = pipeline.fit(data)

## 3. Measure throughput

`Benchmark.throughput` runs `warmup_runs` untimed passes first (so JIT/first-run overhead
doesn't skew the number), then times `trials` more passes and reports the mean rate with a 95%
confidence interval, for every annotation type the pipeline produced.

In [13]:
report = Benchmark.throughput(pipeline_model, data, warmup_runs=1, trials=3)
print(report)

Throughput over 3 trial(s), mean elapsed 2.864 sec/trial
  document             310.5 ± 17.9 items/sec (type: document, n=2664)
  sentence             2,111.5 ± 121.7 items/sec (type: document, n=18114)
  token                39,653.4 ± 2,286.0 items/sec (type: token, n=340173)

## Try it yourself

Swap `pipeline_model` and `data` for your own pipeline and your own dataset -- `Benchmark`
figures out what to measure from what your pipeline actually outputs, no configuration needed.